# Task 4 — Luminance robustness
Zmiany kanału Y (YCbCr): kwadratowe, liniowe (5 współczynników), offset (4 wartości).

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, cv2, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from src.model import get_insightface_model, get_embedding, cosine_similarity
from src.database import EmbeddingDB
from src.degradation import (
    apply_luminance_quadratic,
    apply_luminance_linear, LUMINANCE_LINEAR_COEFFS,
    apply_luminance_offset, LUMINANCE_OFFSETS
)
from src.metrics import compute_far_frr
from src.utils import list_images

app = get_insightface_model('buffalo_l', ctx_id=0)
db = EmbeddingDB.from_file()
THRESHOLD = 0.4
TEST_DIR = Path('../data/test')

def run_experiment(transform, images, label=''):
    scores, labs = [], []
    for user_id, img in images:
        t_img = transform(img)
        emb = get_embedding(app, t_img)
        if emb is None: continue
        refs = db.get_user_embeddings(user_id)
        score = max(cosine_similarity(emb, r) for r in refs)
        scores.append(score); labs.append(1)
    far, frr, acc = compute_far_frr(np.array(scores), np.array(labs), THRESHOLD)
    return {'experiment': label, 'FAR': far, 'FRR': frr, 'Acc': acc, 'n': len(scores)}

In [ ]:
all_images = []
for user_dir in TEST_DIR.iterdir():
    if user_dir.name not in db.get_all_users(): continue
    for p in list_images(user_dir):
        img = cv2.imread(str(p))
        if img is not None: all_images.append((user_dir.name, img))

results = []
# Quadratic
results.append(run_experiment(apply_luminance_quadratic, all_images, 'quadratic'))
# Linear
for c in LUMINANCE_LINEAR_COEFFS:
    results.append(run_experiment(lambda img, c=c: apply_luminance_linear(img, c), all_images, f'linear_{c:.3f}'))
# Offset
for o in LUMINANCE_OFFSETS:
    results.append(run_experiment(lambda img, o=o: apply_luminance_offset(img, o), all_images, f'offset_{o:+d}'))

df = pd.DataFrame(results)
print(df.to_string())
df.to_csv('../results/task4/luminance_results.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
x = range(len(df))
ax.bar(x, df['Acc'], label='Accuracy')
ax.set_xticks(list(x)); ax.set_xticklabels(df['experiment'], rotation=45, ha='right')
ax.set_ylabel('Accuracy'); ax.legend()
plt.tight_layout()
plt.savefig('../results/task4/luminance_plot.png', dpi=150)
plt.show()